In [2]:
import pandas as pd
import numpy as np
import os
import pymrio
import pandas as pd
import numpy as np
import os

In [18]:
def makeFloat(x):
    if isinstance(x, str):
        return float(x.replace(',', ''))
    elif isinstance(x, float) or isinstance(x, int):
        return x
    else:
        return 0.0

### Multiplier matrix (M) calculations

In [ ]:
def calculate_m_matrix_for_year(year,charact_table):
    print(f"Processing year {year}...")
    
    # Load EXIOBASE data for the year
    exio3 = pymrio.parse_exiobase3(path=f"./Data 2010-2020/IOT_{year}_ixi.zip")
    exio3.calc_all()

    A = exio3.A 
    x = exio3.x 
    F = exio3.air_emissions.F
    L = exio3.L

    # Calculate direct emission intensities (S = F / x)
    x_series = x['indout']
    S = F.div(x_series.replace(0, np.nan), axis=1).fillna(0)

    # Calculate multiplier matrix M = S × L
    S_index = S.index 
    L_columns = L.columns  
    S_values = S.to_numpy()
    L_values = L.to_numpy()
    S_values = np.nan_to_num(S_values, nan=0.0, posinf=0.0, neginf=0.0)
    L_values = np.nan_to_num(L_values, nan=0.0, posinf=0.0, neginf=0.0)
    M_values = S_values @ L_values 
    M = pd.DataFrame(M_values, index=S_index, columns=L_columns)

    # Characterize emissions
    characterized_results = {}
    impact_categories = charact_table['characterized_name_column'].unique()
    
    for category in impact_categories:
        characterized_results[category] = np.zeros(M.shape[1])
    
    for i, stressor in enumerate(M.index):
        matches = charact_table[charact_table['names'] == stressor]
        
        if not matches.empty:
            for _, row in matches.iterrows():
                category = row['characterized_name_column']
                factor = row['characterization_factors_column']
                characterized_results[category] += M.iloc[i, :] * factor
    
    characterized_M = pd.DataFrame(characterized_results, index=M.columns).T
    m = characterized_M.transpose()
    
    new_columns = []
    for col in m.columns:
        if isinstance(col, tuple):
            new_columns.append(f"{col[0]} {col[1]}")
        else:
            new_columns.append(col)
    m.columns = new_columns
    
    # Reset index and filter for health sector
    m = m.reset_index()
    m_matrix = m[m['sector'] == 'Health and social work (85)'].copy()
    m_matrix.rename(columns={'GWP 100, AR5': 'GWP 100, AR5 (kg CO2e)'}, inplace=True)
    
    print(f"  Year {year} completed. m_matrix shape: {m_matrix.shape}")
    
    return m_matrix



In [ ]:
#load characterization factors
charact_table = pd.read_csv("./ipcc_gwp_characterization.csv")
charact_table['characterized_unit_column'] = charact_table['characterized_name_column'].str.extract(r'\((.*?)\)')
charact_table['characterized_name_column'] = charact_table['characterized_name_column'].str.replace(r'\s*\(.*?\)', '', regex=True)

#Calculate and save m_matrix for all years (2010-2020)
m_matrices = {}
for year in range(2010, 2021):  # 2010 to 2020 inclusive
    m_matrix = calculate_m_matrix_for_year(year, charact_table)
    m_matrices[year] = m_matrix
    
    # Save m_matrix to CSV
    output_filename = f"./M_matrices/m_matrix_{year}.csv"
    m_matrix.to_csv(output_filename, index=False)
    print(f"  Saved: {output_filename}\n")

### Calcuate Emissions Footprints

In [13]:

def makeFloat(x):
    if isinstance(x, str):
        return float(x.replace(',', ''))
    elif isinstance(x, float) or isinstance(x, int):
        return x
    else:
        return 0.0

def calculate_emissions_for_year(year, ghed_data):
    # Load multiplier matrix
    m_matrix = pd.read_csv(f'./M_matrices/m_matrix_{year}.csv')
    ghed = ghed_data[ghed_data['year'] == year].copy()

    # Initialize lists
    ghg_intensity_values = []
    pm_values = []
    o3_values = []

    for index, row in ghed.iterrows():
        matching_row = m_matrix[m_matrix['region'] == row['Alpha-2 code']] 
        if not matching_row.empty:
            ghg_intensity_values.append(matching_row['GWP 100, AR5 (kg CO2e)'].iloc[0])
            pm_values.append(matching_row['Particulate matter formation, Disability Adjusted Life Years '].iloc[0])
            o3_values.append(matching_row['Photochemical ozone formation, Disability Adjusted Life Years '].iloc[0])
        else:
            if row['region'] == 'AFR':
                region_code = 'WF'
            elif row['region'] == 'EUR':
                region_code = 'WE'
            elif row['region'] == 'AMR':
                region_code = 'WL'
            elif row['region'] == 'EMR':
                region_code = 'WM'
            elif row['region'] in ['SEAR', 'WPR']:
                region_code = 'WA'
            else:
                region_code = None

            if region_code and not m_matrix[m_matrix['region'] == region_code].empty:
                ghg_intensity_values.append(m_matrix[m_matrix['region'] == region_code]['GWP 100, AR5 (kg CO2e)'].iloc[0])
                pm_values.append(m_matrix[m_matrix['region'] == region_code]['Particulate matter formation, Disability Adjusted Life Years '].iloc[0])
                o3_values.append(m_matrix[m_matrix['region'] == region_code]['Photochemical ozone formation, Disability Adjusted Life Years '].iloc[0])
            else:
                ghg_intensity_values.append(None)
                pm_values.append(None)
                o3_values.append(None)

    ghed['GHG Emissions intensity (kg CO2e/Million EUR)'] = ghg_intensity_values
    ghed['Health care DALYs PM2.5/MillEUR'] = pm_values
    ghed['Health care DALYs O3/MillEUR'] = o3_values

    econ_adjust = pd.read_csv("./Econ_adjustment_factor.csv", encoding='latin-1')
    econ_adjustment = econ_adjust.loc[econ_adjust['ï»¿Year'] == year, 'Factor'].values[0]

    # Calculations
    health_ghgs_pc = ghed['che_pc_usd'].apply(makeFloat).mul(ghed['GHG Emissions intensity (kg CO2e/Million EUR)'].apply(makeFloat)).mul(econ_adjustment).div(1000000)
    ghed['Health care GHGs kg CO2e per cap, (in mil $)'] = health_ghgs_pc

    health_ghgs = (ghed['Health care GHGs kg CO2e per cap, (in mil $)'].apply(makeFloat).mul(ghed['pop']).mul(1000)).div(1000000000)
    dalys_pm = ghed['che_usd'].apply(makeFloat).mul(ghed['Health care DALYs PM2.5/MillEUR'].apply(makeFloat)).mul(econ_adjustment)
    dalys_o3 = ghed['che_usd'].apply(makeFloat).mul(ghed['Health care DALYs O3/MillEUR'].apply(makeFloat)).mul(econ_adjustment)
    dalys_total = dalys_pm + dalys_o3

    ghed['Health care GHGs Mt'] = health_ghgs
    ghed['Health care DALYs PM2.5'] = dalys_pm
    ghed['Healthcare DALYs O3'] = dalys_o3
    ghed['Healthcare DALYs Total'] = dalys_total

    return ghed


In [ ]:
#Load GHED data (most recent version downloaded on january 2026)
ghed_data = pd.read_csv("./GHED_data_1_20_26_download.csv", encoding='latin-1', thousands=',')
ghed_data = ghed_data[['ï»¿location', 'code', 'region', 'income', 'year', 'che_pc_usd', 'che_usd', 'pop', 'xrt','gdpd']]
ghed_data = ghed_data.rename(columns={'ï»¿location': 'location'})
#import iso country codes to match with GHED data
country_codes = pd.read_csv('./iso_country_codes.csv')
ghed_data = ghed_data.merge(country_codes[['Alpha-3 code', 'Alpha-2 code']], left_on='code', right_on='Alpha-3 code', how='left')

# Loop over years and save results
for year in range(2010, 2021):
    result = calculate_emissions_for_year(year, ghed_data)
    # Create a dictionary to store results for each year
    if 'results_dict' not in locals():
        results_dict = {}
    results_dict[year] = result

# After the loop, save all sheets to a single Excel workbook
with pd.ExcelWriter('RESULTS/3.9.6_emissions_AND_dalys_calcs_all_years.xlsx', engine='xlsxwriter') as writer:
    for yr, df in results_dict.items():
        df.to_excel(writer, sheet_name=str(yr), index=False)
  

### Adjusting and calculating 2021-2023 impacts (after latest model year - 2020)

In [ ]:
#Deflate expenditures for 2021-2023
def deflate_expenditures(ghed_data, year):
    
    ghed_2020 = ghed_data[ghed_data['year'] == 2020].copy()
    ghed = ghed_data[ghed_data['year'] == year].copy()

    # Merge to align countries - using 'left' to keep all countries from selected year
    # Add suffix to distinguish 2020 values
    ghed = ghed.merge(
        ghed_2020[['code', 'gdpd', 'xrt']], 
        on='code', 
        how='left',
        suffixes=('', '_2020')
    )
    
    # Convert from USD to local currency
    ghed['che_pc_local'] = ghed['che_pc_usd'] * ghed['xrt']
    ghed['che_local'] = ghed['che_usd'] * ghed['xrt']
    
    # Convert nominal year to 2020 dollars using GDP Price Index (GDPD)
    # Now this uses the correctly aligned 2020 values
    ghed['gdp_multiplier'] = ghed['gdpd_2020'] / ghed['gdpd']

    # Deflate expenditures to 2020 local currency
    ghed['che_pc_local_2020'] = ghed['che_pc_local'] * ghed['gdp_multiplier']
    ghed['che_local_2020'] = ghed['che_local'] * ghed['gdp_multiplier']
    
    # Convert from 2020 local currency to 2020 USD using exchange rates
    ghed['che_pc_usd_2020'] = ghed['che_pc_local_2020'] / ghed['xrt_2020']
    ghed['che_usd_2020'] = ghed['che_local_2020'] / ghed['xrt_2020']
    
    return ghed

In [18]:
#Calcuate deflated expenditures for all years and save to CSV
for year in range(2021, 2024):
    result = deflate_expenditures(ghed_data, year)
    result.to_csv(f"./deflated_expenditures/ghed_{year}.csv", index=False)
    
   

In [19]:
#EMISSIONS CALCS

def makeFloat(x):
    if isinstance(x, str):
        return float(x.replace(',', ''))
    elif isinstance(x, float) or isinstance(x, int):
        return x
    else:
        return 0.0

def calculate_emissions_for_2020(year, ghed_data):
    # Load multiplier matrix
    m_matrix = pd.read_csv(f'./M_matrices/m_matrix_2020.csv')
    ghed = pd.read_csv(f"./deflated_expenditures/ghed_{year}.csv")

    # Initialize lists
    ghg_intensity_values = []
    pm_values = []
    o3_values = []

    for index, row in ghed.iterrows():
        matching_row = m_matrix[m_matrix['region'] == row['Alpha-2 code']] 
        if not matching_row.empty:
            ghg_intensity_values.append(matching_row['GWP 100, AR5 (kg CO2e)'].iloc[0])
            pm_values.append(matching_row['Particulate matter formation, Disability Adjusted Life Years '].iloc[0])
            o3_values.append(matching_row['Photochemical ozone formation, Disability Adjusted Life Years '].iloc[0])
        else:
            if row['region'] == 'AFR':
                region_code = 'WF'
            elif row['region'] == 'EUR':
                region_code = 'WE'
            elif row['region'] == 'AMR':
                region_code = 'WL'
            elif row['region'] == 'EMR':
                region_code = 'WM'
            elif row['region'] in ['SEAR', 'WPR']:
                region_code = 'WA'
            else:
                region_code = None

            if region_code and not m_matrix[m_matrix['region'] == region_code].empty:
                ghg_intensity_values.append(m_matrix[m_matrix['region'] == region_code]['GWP 100, AR5 (kg CO2e)'].iloc[0])
                pm_values.append(m_matrix[m_matrix['region'] == region_code]['Particulate matter formation, Disability Adjusted Life Years '].iloc[0])
                o3_values.append(m_matrix[m_matrix['region'] == region_code]['Photochemical ozone formation, Disability Adjusted Life Years '].iloc[0])
            else:
                ghg_intensity_values.append(None)
                pm_values.append(None)
                o3_values.append(None)

    ghed['GHG Emissions intensity (kg CO2e/Million EUR)'] = ghg_intensity_values
    ghed['Health care DALYs PM2.5/MillEUR'] = pm_values
    ghed['Health care DALYs O3/MillEUR'] = o3_values

    econ_adjust = pd.read_csv("./Econ_adjustment_factor.csv", encoding='latin-1')
    econ_adjustment = econ_adjust.loc[econ_adjust['ï»¿Year'] == 2020, 'Factor'].values[0]

    # Calculations
    health_ghgs_pc = ghed['che_pc_usd_2020'].apply(makeFloat).mul(ghed['GHG Emissions intensity (kg CO2e/Million EUR)'].apply(makeFloat)).mul(econ_adjustment).div(1000000)
    ghed['Health care GHGs kg CO2e per cap, (in mil $)'] = health_ghgs_pc

    health_ghgs = (ghed['Health care GHGs kg CO2e per cap, (in mil $)'].apply(makeFloat).mul(ghed['pop']).mul(1000)).div(1000000000)
    dalys_pm = ghed['che_usd_2020'].apply(makeFloat).mul(ghed['Health care DALYs PM2.5/MillEUR'].apply(makeFloat)).mul(econ_adjustment)
    dalys_o3 = ghed['che_usd_2020'].apply(makeFloat).mul(ghed['Health care DALYs O3/MillEUR'].apply(makeFloat)).mul(econ_adjustment)
    dalys_total = dalys_pm + dalys_o3

    ghed['Health care GHGs Mt'] = health_ghgs
    ghed['Health care DALYs PM2.5'] = dalys_pm
    ghed['Healthcare DALYs O3'] = dalys_o3
    ghed['Healthcare DALYs Total'] = dalys_total

    return ghed

# save each year as a separate CSV
os.makedirs('RESULTS', exist_ok=True)
for year in range(2021, 2024):
    result = calculate_emissions_for_2020(year, ghed_data)
    output_path = f'RESULTS/unadjusted_emissions_{year}.csv'
    result.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")
  


Saved: RESULTS/unadjusted_emissions_2021.csv
Saved: RESULTS/unadjusted_emissions_2022.csv
Saved: RESULTS/unadjusted_emissions_2023.csv


In [ ]:
#Load primap data
primap_data = pd.read_csv("./Guetschow_et_al_2025a-PRIMAP-hist_v2.7_final_22-Aug-2025.csv")

#Update emissions funtion

def update_health_emissions(ghed_data, primap_data, base_year, update_year):
   
    # Copy of the input data 
    ghed_updated = pd.read_csv(f"./RESULTS/unadjusted_emissions_{update_year}.csv") 

    primap_filtered = primap_data[
        (primap_data['entity'] == 'KYOTOGHG (AR5GWP100)') & 
        (primap_data['category (IPCC2006_PRIMAP)'] == '5') &
        (primap_data['scenario (PRIMAP-hist)'] == 'HISTCR')
    ]

    # Get base year and update year values and calculate growth rates
    growth_rates = {}

    for _, row in primap_filtered.iterrows():
        country_code = row['area (ISO3)']

        # Extract values for both years if they exist
        if str(base_year) in primap_filtered.columns and str(update_year) in primap_filtered.columns:
            base_value = row[str(base_year)]
            update_value = row[str(update_year)]

            # Calc growth rate
            if base_value > 0:  # Avoid division by zero
                growth_rate = (update_value - base_value) / base_value
            else:
                growth_rate = 0

            growth_rates[country_code] = growth_rate

    # List of columns to apply the growth rate
    columns_to_update = [
        'Health care GHGs Mt',
        'Health care GHGs kg CO2e per cap, (in mil $)',
        'Health care DALYs PM2.5',
        'Healthcare DALYs O3',
        'Healthcare DALYs Total'
    ]

    # Apply growth rates to each country in the GHED data
    for i, row in ghed_updated.iterrows():
        country_code = row['code']

        if country_code in growth_rates:
            growth_rate = growth_rates[country_code]
            ghed_updated.at[i, f'Growth rate ({base_year}-{update_year})'] = growth_rate

            for col in columns_to_update:
                if col in ghed_updated.columns:
                    updated_value = row[col] * (1 + growth_rate)
                    ghed_updated.at[i, f'{col} ({update_year})'] = updated_value
                else:
                    print(f"Warning: Column {col} not found in dataset")

        else:
            print(f"Warning: No growth rate found for {country_code}, using original values")
            for col in columns_to_update:
                if col in ghed_updated.columns:
                    ghed_updated.at[i, f'{col} ({update_year})'] = row[col]
        
    return ghed_updated


for update_year in range(2021, 2024):
    result = update_health_emissions(ghed_data, primap_data, 2020, update_year)
    output_path = f'RESULTS/primap_adjusted_emissions_{update_year}.csv'
    result.to_csv(output_path, index=False)





#### 2023 by scope

In [ ]:
# Scope 1 / 2 / 3 disaggregation for Health sector, 2020

#load characterization factors
charact_table = pd.read_csv("./ipcc_gwp_characterization.csv")
charact_table['characterized_unit_column'] = charact_table['characterized_name_column'].str.extract(r'\((.*?)\)')
charact_table['characterized_name_column'] = charact_table['characterized_name_column'].str.replace(r'\s*\(.*?\)', '', regex=True)

year = 2020
target_sector = 'Health and social work (85)'
target_impact = 'GWP 100, AR5'  # must match your charact_table after unit strip

exio3 = pymrio.parse_exiobase3(path=f"./Data 2010-2020/IOT_{year}_ixi.zip")
exio3.calc_all()

A = exio3.A
x = exio3.x['indout']
F = exio3.air_emissions.F
L = exio3.L

# Build characterized S matrix 
S_raw = F.div(x.replace(0, np.nan), axis=1).fillna(0)

impact_categories = charact_table['characterized_name_column'].unique()
characterized_S = {cat: np.zeros(S_raw.shape[1]) for cat in impact_categories}

for i, stressor in enumerate(S_raw.index):
    matches = charact_table[charact_table['names'] == stressor]
    if not matches.empty:
        for _, row in matches.iterrows():
            characterized_S[row['characterized_name_column']] += S_raw.iloc[i, :] * row['characterization_factors_column']

S_char = pd.DataFrame(characterized_S, index=S_raw.columns).T

#Build characterized M = S_char @ L 
S_vals = np.nan_to_num(S_char.to_numpy(), nan=0.0, posinf=0.0, neginf=0.0)
L_vals = np.nan_to_num(L.to_numpy(), nan=0.0, posinf=0.0, neginf=0.0)
M_char = pd.DataFrame(S_vals @ L_vals, index=S_char.index, columns=L.columns)

# Identify electricity sectors 
elec_sectors = [c for c in A.columns if 'electricity' in c[1].lower()]

# Scope 2: S_char[elec_sectors] @ A[elec_sectors]
S_elec_vals = np.nan_to_num(S_char[elec_sectors].to_numpy(), nan=0.0, posinf=0.0, neginf=0.0)
A_elec_vals = np.nan_to_num(A.loc[elec_sectors].to_numpy(), nan=0.0, posinf=0.0, neginf=0.0)
scope2_char = pd.DataFrame(S_elec_vals @ A_elec_vals, index=S_char.index, columns=A.columns)

# Scope 1, 3, Total 
scope1_char = S_char
scope3_char = M_char - scope1_char - scope2_char

# Filter to health sector GWP100 row across all regions
def extract_health_row(df):
    row = df.loc[target_impact]
    health_cols = [c for c in df.columns if c[1] == target_sector]
    result = row[health_cols]
    result.index = [c[0] for c in result.index]  
    return result

s1 = extract_health_row(scope1_char).rename('Scope 1')
s2 = extract_health_row(scope2_char).rename('Scope 2')
s3 = extract_health_row(scope3_char).rename('Scope 3')
total = extract_health_row(M_char).rename('Total lifecycle')

scopes_2020 = pd.concat([s1, s2, s3, total], axis=1)
scopes_2020.index.name = 'region'
scopes_2020['year'] = year

scopes_2020.to_csv('./RESULTS/impacts_by_scopes_2020.csv')

In [10]:
scopes_2020 = pd.read_csv('./RESULTS/impacts_by_scopes_2020.csv')
scopes_2020.head()

,region,Scope 1,Scope 2,Scope 3,Total lifecycle,year
0,AT,41056.818992,1215.649911,103611.929333,145884.398236,2020
1,BE,10235.010570,3541.260517,125323.554069,139099.825156,2020
2,BG,59630.262916,80802.983579,298953.715431,439386.961926,2020
3,CY,20286.688231,26889.529683,309636.051052,356812.268966,2020
4,CZ,8330.584966,25829.391008,127147.906188,161307.882162,2020


##### CALCULATE EMISSIONS BY SCOPE - USING DEFLATED 2023 EXPENDITURES AND ADJUSTED WITH PRIMAP GROWTH RATES

In [19]:
#Unadjusted 2023 GHG emissions by scope, using 2020 model and deflated exp
update_year = 2023
ghed_2023 = pd.read_csv(f"./deflated_expenditures/ghed_{update_year}.csv")
econ_adjust = pd.read_csv("./Econ_adjustment_factor.csv", encoding='latin-1')
econ_adjustment = econ_adjust.loc[econ_adjust['ï»¿Year'] == 2020, 'Factor'].values[0]

# Rename index column to 'Alpha-2 code' to match GHED directly, avoiding region clash
scopes_reset = scopes_2020.reset_index().rename(columns={'region': 'Alpha-2 code'})

ghed_2023 = ghed_2023.merge(
    scopes_reset[['Alpha-2 code', 'Scope 1', 'Scope 2', 'Scope 3', 'Total lifecycle']],
    on='Alpha-2 code',
    how='left'
)

#For unmatched countries, fall back to RoW aggregates
row_map = {'AFR': 'WF', 'EUR': 'WE', 'AMR': 'WL', 'EMR': 'WM', 'SEAR': 'WA', 'WPR': 'WA'}

for scope_col in ['Scope 1', 'Scope 2', 'Scope 3', 'Total lifecycle']:
    for i, row in ghed_2023.iterrows():
        if pd.isna(row[scope_col]):
            row_code = row_map.get(row['region'], None)
            if row_code and row_code in scopes_reset['Alpha-2 code'].values:
                ghed_2023.at[i, scope_col] = scopes_reset.loc[
                    scopes_reset['Alpha-2 code'] == row_code, scope_col].values[0]

#Calculate total GHG emissions (Mt CO2e) by scope
for scope_col in ['Scope 1', 'Scope 2', 'Scope 3', 'Total lifecycle']:
    col_label = f'GHGs Mt {scope_col} 2023 (unadjusted)'
    ghed_2023[col_label] = (
        ghed_2023[scope_col].apply(makeFloat)
        .mul(ghed_2023['che_usd_2020'].apply(makeFloat))
        .mul(econ_adjustment)
        .div(1e9)
    )

os.makedirs('RESULTS', exist_ok=True)
ghed_2023.to_csv('./RESULTS/unadjusted_scope_emissions_2023.csv', index=False)

In [ ]:

#PRIMAP-adjusted 2023 GHG emissions by scope - applies country-level growth rate (2020→2023)
base_year = 2020
ghed_2023_adj = pd.read_csv('./RESULTS/unadjusted_scope_emissions_2023.csv')
primap_data = pd.read_csv("./Guetschow_et_al_2025a-PRIMAP-hist_v2.7_final_22-Aug-2025.csv")

#Filter PRIMAP
primap_filtered = primap_data[
    (primap_data['entity'] == 'KYOTOGHG (AR5GWP100)') &
    (primap_data['category (IPCC2006_PRIMAP)'] == '5') &
    (primap_data['scenario (PRIMAP-hist)'] == 'HISTCR')
]

#country-level growth rate
growth_rates = {}
for _, row in primap_filtered.iterrows():
    country_code = row['area (ISO3)']
    if str(base_year) in primap_filtered.columns and str(update_year) in primap_filtered.columns:
        base_val = row[str(base_year)]
        update_val = row[str(update_year)]
        if pd.notna(base_val) and pd.notna(update_val) and base_val > 0:
            growth_rates[country_code] = (update_val - base_val) / base_val
        else:
            growth_rates[country_code] = 0

#Apply growth rate to each scope column
scope_unadjusted_cols = {
    'Scope 1': 'GHGs Mt Scope 1 2023 (unadjusted)',
    'Scope 2': 'GHGs Mt Scope 2 2023 (unadjusted)',
    'Scope 3': 'GHGs Mt Scope 3 2023 (unadjusted)',
    'Total lifecycle': 'GHGs Mt Total lifecycle 2023 (unadjusted)'
}

for scope_label, unadj_col in scope_unadjusted_cols.items():
    adj_col = f'GHGs Mt {scope_label} 2023 (PRIMAP adjusted)'
    ghed_2023_adj[adj_col] = None

    for i, row in ghed_2023_adj.iterrows():
        country_iso3 = row['code']
        base_emission = row[unadj_col]

        if country_iso3 in growth_rates and pd.notna(base_emission):
            growth_rate = growth_rates[country_iso3]
            ghed_2023_adj.at[i, adj_col] = float(base_emission) * (1 + growth_rate)
        else:
            # No PRIMAP match — retain unadjusted value and flag it
            ghed_2023_adj.at[i, adj_col] = base_emission
            if i == 0:  # only warn once per missing country to avoid noise
                print(f"Warning: No PRIMAP growth rate found for {country_iso3}, using unadjusted value")

# Add growth rate column for transparency
ghed_2023_adj['PRIMAP growth rate (2020-2023)'] = ghed_2023_adj['code'].map(growth_rates)

adjusted_cols = ['location', 'code',
                 'GHGs Mt Scope 1 2023 (PRIMAP adjusted)',
                 'GHGs Mt Scope 2 2023 (PRIMAP adjusted)',
                 'GHGs Mt Scope 3 2023 (PRIMAP adjusted)',
                 'GHGs Mt Total lifecycle 2023 (PRIMAP adjusted)',
                 'PRIMAP growth rate (2020-2023)']

ghed_2023_adj.to_csv('./RESULTS/primap_adjusted_scope_emissions_2023.csv', index=False)


#### years by in region & out of region calcs

In [ ]:
#Build within/outside region M matrices (2020)

year = 2020
target_sector = 'Health and social work (85)'
target_impact = 'GWP 100, AR5'

exio3 = pymrio.parse_exiobase3(path=f"./Data 2010-2020/IOT_{year}_ixi.zip")
exio3.calc_all()

A = exio3.A
x = exio3.x['indout']
F = exio3.air_emissions.F
L = exio3.L

S_raw = F.div(x.replace(0, np.nan), axis=1).fillna(0)
S_raw_vals = np.nan_to_num(S_raw.to_numpy(), nan=0.0, posinf=0.0, neginf=0.0)
L_vals = np.nan_to_num(L.to_numpy(), nan=0.0, posinf=0.0, neginf=0.0)

# Full M in raw stressor space: [n_stressors x n_sectors_all_regions]
M_raw = pd.DataFrame(S_raw_vals @ L_vals, index=S_raw.index, columns=L.columns)

# Get unique consuming regions (from column MultiIndex)
consuming_regions = L.columns.get_level_values(0).unique()

# For each consuming region, mask M columns to within/outside emitting region
# M rows are stressors, columns are (emitting_region, sector)
#Identify emitting region from column level 0

def characterize(matrix_vals, col_index):
    """Apply GWP100 characterization to a raw stressor matrix."""
    result = np.zeros(matrix_vals.shape[1])
    for i, stressor in enumerate(S_raw.index):
        matches = charact_table[charact_table['names'] == stressor]
        if not matches.empty:
            gwp_row = matches[matches['characterized_name_column'] == target_impact]
            if not gwp_row.empty:
                result += matrix_vals[i, :] * gwp_row['characterization_factors_column'].values[0]
    return pd.Series(result, index=col_index)

# Build within/outside characterized multipliers for health sector, per consuming region
records = []
health_cols = [c for c in L.columns if c[1] == target_sector]

for consuming_region in consuming_regions:
    health_col = (consuming_region, target_sector)
    if health_col not in L.columns:
        continue

    col_idx = L.columns.get_loc(health_col)
    M_col = S_raw_vals @ L_vals[:, col_idx]  

    # Split by emitting region
    emitting_regions = S_raw.columns.get_level_values(0)  # stressor rows indexed by (em_region, sector)
    # S_raw rows = stressors (no region), columns = (region, sector)
    # M_raw rows = stressors, columns = (consuming_region, sector)
    # To split by emitting region, need S diagonalized — use S_raw columns to identify emitting region

    within_mask = np.array([reg == consuming_region for reg in L.index.get_level_values(0)])
    outside_mask = ~within_mask

    # M_within for this consuming region = S_raw @ (L_col masked to within-region rows)
    L_col = L_vals[:, col_idx]
    M_col_within = S_raw_vals @ (L_col * within_mask)
    M_col_outside = S_raw_vals @ (L_col * outside_mask)

    # Characterize both
    def char_vector(vec):
        result = 0.0
        for i, stressor in enumerate(S_raw.index):
            matches = charact_table[
                (charact_table['names'] == stressor) &
                (charact_table['characterized_name_column'] == target_impact)
            ]
            if not matches.empty:
                result += vec[i] * matches['characterization_factors_column'].values[0]
        return result

    records.append({
        'region': consuming_region,
        'Within region': char_vector(M_col_within),
        'Outside region': char_vector(M_col_outside),
        'Total': char_vector(M_col_within + M_col_outside)
    })

within_outside_2020 = pd.DataFrame(records).set_index('region')
print(within_outside_2020.head())

        Within region  Outside region          Total
region                                              
AT       85086.769521    60797.628715  145884.398236
BE       57072.315862    82027.509295  139099.825156
BG      294016.964705   145369.997221  439386.961926
CY      144067.004238   212745.264727  356812.268966
CZ       79598.220009    81709.662153  161307.882162


In [26]:
# Unadjusted 2023 within/outside GHG emissions
update_year = 2023

ghed_2023 = pd.read_csv(f"./deflated_expenditures/ghed_{update_year}.csv")
econ_adjust = pd.read_csv("./Econ_adjustment_factor.csv", encoding='latin-1')
econ_adjustment = econ_adjust.loc[econ_adjust['ï»¿Year'] == 2020, 'Factor'].values[0]

wo_reset = within_outside_2020.reset_index().rename(columns={'region': 'Alpha-2 code'})
ghed_2023 = ghed_2023.merge(
    wo_reset[['Alpha-2 code', 'Within region', 'Outside region', 'Total']],
    on='Alpha-2 code', how='left'
)

row_map = {'AFR': 'WF', 'EUR': 'WE', 'AMR': 'WL', 'EMR': 'WM', 'SEAR': 'WA', 'WPR': 'WA'}
for col in ['Within region', 'Outside region', 'Total']:
    for i, row in ghed_2023.iterrows():
        if pd.isna(row[col]):
            row_code = row_map.get(row['region'], None)
            if row_code and row_code in wo_reset['Alpha-2 code'].values:
                ghed_2023.at[i, col] = wo_reset.loc[wo_reset['Alpha-2 code'] == row_code, col].values[0]

for col in ['Within region', 'Outside region', 'Total']:
    ghed_2023[f'GHGs Mt {col} 2023 (unadjusted)'] = (
        ghed_2023[col].apply(makeFloat)
        .mul(ghed_2023['che_usd_2020'].apply(makeFloat))
        .mul(econ_adjustment)
        .div(1e9)
    )

ghed_2023.to_csv('./RESULTS/unadjusted_within_outside_emissions_2023.csv', index=False)


In [27]:
#PRIMAP-adjusted 2023 within/outside GHG emissions
ghed_2023_adj = pd.read_csv('./RESULTS/unadjusted_within_outside_emissions_2023.csv')

primap_filtered = primap_data[
    (primap_data['entity'] == 'KYOTOGHG (AR5GWP100)') &
    (primap_data['category (IPCC2006_PRIMAP)'] == '5') &
    (primap_data['scenario (PRIMAP-hist)'] == 'HISTCR')
]

growth_rates = {}
for _, row in primap_filtered.iterrows():
    base_val = row[str(2020)]
    update_val = row[str(update_year)]
    if pd.notna(base_val) and pd.notna(update_val) and base_val > 0:
        growth_rates[row['area (ISO3)']] = (update_val - base_val) / base_val

unadj_cols = {
    'Within region': 'GHGs Mt Within region 2023 (unadjusted)',
    'Outside region': 'GHGs Mt Outside region 2023 (unadjusted)',
    'Total':          'GHGs Mt Total 2023 (unadjusted)'
}

for label, unadj_col in unadj_cols.items():
    adj_col = f'GHGs Mt {label} 2023 (PRIMAP adjusted)'
    ghed_2023_adj[adj_col] = ghed_2023_adj.apply(
        lambda row: float(row[unadj_col]) * (1 + growth_rates[row['code']])
        if row['code'] in growth_rates and pd.notna(row[unadj_col])
        else row[unadj_col],
        axis=1
    )

ghed_2023_adj['PRIMAP growth rate (2020-2023)'] = ghed_2023_adj['code'].map(growth_rates)

ghed_2023_adj.to_csv('./RESULTS/primap_adjusted_within_outside_emissions_2023.csv', index=False)
